In [ ]:
#1
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
#2
!pip install statsmodels dagshub mlflow -q

import warnings, logging
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
import mlflow
import mlflow.statsmodels
import dagshub
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, RegressorMixin

warnings.filterwarnings('ignore')
logging.getLogger('statsmodels').setLevel(logging.ERROR)

dagshub.init(repo_owner='lshek22', repo_name='walmart-recruiting-store-sales-forecasting', mlflow=True)
mlflow.set_experiment("SARIMA_Training")
print("Ready.")

In [ ]:
#3
DATA_DIR = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting"

train    = pd.read_csv(f"{DATA_DIR}/train.csv.zip")
test     = pd.read_csv(f"{DATA_DIR}/test.csv.zip")
stores   = pd.read_csv(f"{DATA_DIR}/stores.csv")
features = pd.read_csv(f"{DATA_DIR}/features.csv.zip")

train['Date'] = pd.to_datetime(train['Date'])
test['Date']  = pd.to_datetime(test['Date'])
features['Date'] = pd.to_datetime(features['Date'])

train = train.merge(stores, on='Store', how='left')
train = train.merge(features.drop(columns=['IsHoliday']), on=['Store','Date'], how='left')

test = test.merge(stores, on='Store', how='left')
test = test.merge(features.drop(columns=['IsHoliday']), on=['Store','Date'], how='left')

md_cols = ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']
train[md_cols] = train[md_cols].fillna(0)
test[md_cols]  = test[md_cols].fillna(0)

train['Weekly_Sales'] = train['Weekly_Sales'].clip(lower=0)

train = train.sort_values(['Store','Dept','Date']).reset_index(drop=True)
test  = test.sort_values(['Store','Dept','Date']).reset_index(drop=True)

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")
print(f"Unique series: {train.groupby(['Store','Dept']).ngroups}")

## SARIMA theory check (weekly seasonality)

SARIMA extends ARIMA with seasonal terms `(P, D, Q, s)`. For Walmart weekly sales, `s=52`.
We first inspect one representative series (Store 1, Dept 1) for trend + yearly seasonality.

In [ ]:
#4
store, dept = 1, 1
s1d1 = train[(train['Store']==store) & (train['Dept']==dept)].set_index('Date')['Weekly_Sales']

s1d1_seasonal_diff = s1d1.diff(52).dropna()
s1d1_both_diff = s1d1_seasonal_diff.diff().dropna()

def check_stationarity(series, series_name="Series"):
    clean = series.dropna()
    result = adfuller(clean, autolag='AIC')
    return {
        'series': series_name,
        'adf_stat': result[0],
        'p_value': result[1],
        'is_stationary': result[1] < 0.05,
        'n_obs': len(clean),
    }

raw_stat = check_stationarity(s1d1, "Raw sales")
seasonal_diff_stat = check_stationarity(s1d1_seasonal_diff, "Seasonal diff (lag 52)")
both_diff_stat = check_stationarity(s1d1_both_diff, "Seasonal + first diff")

print("Stationarity checks:")
print(f"raw sales                 p={raw_stat['p_value']:.4f} stationary={raw_stat['is_stationary']}")
print(f"seasonal diff (52)        p={seasonal_diff_stat['p_value']:.4f} stationary={seasonal_diff_stat['is_stationary']}")
print(f"seasonal + first diff     p={both_diff_stat['p_value']:.4f} stationary={both_diff_stat['is_stationary']}")

In [ ]:
#5
fig, axes = plt.subplots(3, 2, figsize=(15, 11))
fig.patch.set_facecolor('#0f1117')
for ax in axes.flatten():
    ax.set_facecolor('#1a1d27')
    ax.tick_params(colors='#8b949e')
    for spine in ax.spines.values():
        spine.set_color('#3a3d4d')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[0,0].plot(s1d1.index, s1d1.values, color='#58a6ff', linewidth=1.5)
axes[0,0].set_title('Raw Weekly Sales', color='white', fontsize=11)
axes[0,0].set_ylabel('Weekly Sales', color='#8b949e')

axes[0,1].plot(s1d1_seasonal_diff.index, s1d1_seasonal_diff.values, color='#3fb950', linewidth=1.2)
axes[0,1].axhline(0, color='white', alpha=0.3)
axes[0,1].set_title('After seasonal differencing (lag=52)', color='white', fontsize=11)

# statsmodels constraint: nlags must be < 50% of sample size
max_safe_nlags = max(1, (len(s1d1_both_diff) // 2) - 1)
nlags = min(60, max_safe_nlags)
print(f"ACF/PACF using nlags={nlags} (len(diffed)={len(s1d1_both_diff)})")

acf_vals = acf(s1d1_both_diff, nlags=nlags, fft=True)
pacf_vals = pacf(s1d1_both_diff, nlags=nlags)
axes[1,0].bar(range(len(acf_vals)), acf_vals, color='#58a6ff', alpha=0.8, width=0.6)
axes[1,0].axhline(0, color='white', alpha=0.3)
axes[1,0].set_title('ACF after seasonal+first diff', color='white', fontsize=11)
axes[1,1].bar(range(len(pacf_vals)), pacf_vals, color='#bc8cff', alpha=0.8, width=0.6)
axes[1,1].axhline(0, color='white', alpha=0.3)
axes[1,1].set_title('PACF after seasonal+first diff', color='white', fontsize=11)

decomp = seasonal_decompose(s1d1, model='additive', period=52)
axes[2,0].plot(decomp.trend.index, decomp.trend.values, color='#d29922', linewidth=1.5)
axes[2,0].set_title('Trend component', color='white', fontsize=11)
axes[2,1].plot(decomp.seasonal.index, decomp.seasonal.values, color='#3fb950', linewidth=1.2)
axes[2,1].set_title('Seasonal component (period=52)', color='white', fontsize=11)

for ax in axes.flatten():
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('SARIMA prerequisites: seasonality and decomposition', color='white', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('sarima_educational_analysis.png', dpi=130, bbox_inches='tight', facecolor='#0f1117')
plt.show()

with mlflow.start_run(run_name='SARIMA_Cleaning'):
    mlflow.log_params({
        'markdown_imputation': 'fill_zero',
        'negative_sales': 'clip_to_zero',
        'seasonal_period': 52,
        'validation_weeks': 8,
        'note': 'classical model uses raw weekly series, no lag features',
    })
    mlflow.log_artifact('sarima_educational_analysis.png', 'plots')
    print('Cleaning / EDA run logged')

In [ ]:
#6
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

SPLIT_DATE = train['Date'].max() - pd.Timedelta(weeks=8)

train_set = train[train['Date'] <= SPLIT_DATE].copy()
val_set   = train[train['Date'] >  SPLIT_DATE].copy()

print(f"Train period: {train_set['Date'].min().date()} -> {train_set['Date'].max().date()}")
print(f"Val period  : {val_set['Date'].min().date()}   -> {val_set['Date'].max().date()}")
print(f"Val weeks   : {val_set['Date'].nunique()}")

In [ ]:
#7
series = (train_set[(train_set['Store']==store) & (train_set['Dept']==dept)]
          .set_index('Date')['Weekly_Sales']
          .asfreq('W-FRI'))

val_series = (val_set[(val_set['Store']==store) & (val_set['Dept']==dept)]
              .set_index('Date')['Weekly_Sales'])

SEASONAL_PERIOD = 52
BASE_ORDER = (1, 1, 1)
BASE_SEASONAL = (1, 1, 1, SEASONAL_PERIOD)

model = SARIMAX(series, order=BASE_ORDER, seasonal_order=BASE_SEASONAL,
                enforce_stationarity=False, enforce_invertibility=False)
result = model.fit(disp=False)
print(result.summary())

forecast = result.forecast(steps=len(val_series))
score = wmae(
    val_series.values,
    forecast.values,
    val_set[(val_set['Store']==store) & (val_set['Dept']==dept)]['IsHoliday'].values,
)
print(f"SARIMA{BASE_ORDER}x{BASE_SEASONAL} WMAE on Store {store} Dept {dept}: {score:,.2f}")

fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d27')
ax.plot(series.index, series.values, color='#58a6ff', linewidth=1.5, label='Training data')
ax.plot(val_series.index, val_series.values, color='#3fb950', linewidth=1.5, label='Actual (validation)')
ax.plot(val_series.index, forecast.values, color='#f78166', linewidth=1.5, linestyle='--', label='SARIMA forecast')
ax.axvline(SPLIT_DATE, color='white', alpha=0.4, linewidth=1, linestyle=':')
ax.set_title(f'SARIMA baseline forecast - Store {store} Dept {dept}  WMAE={score:,.0f}', color='white')
ax.legend(framealpha=0.3)
plt.tight_layout()
plt.savefig('sarima_single_series_forecast.png', dpi=130, facecolor='#0f1117')
plt.show()

In [ ]:
#8
ORDERS_TO_TEST = {
  # original configs
    'SARIMA_111_011_52': ((1,1,1), (0,1,1,52)),
    'SARIMA_111_110_52': ((1,1,1), (1,1,0,52)),
    'SARIMA_111_111_52': ((1,1,1), (1,1,1,52)),
    'SARIMA_011_011_52': ((0,1,1), (0,1,1,52)),
    # new configs: simpler seasonal structure (often generalizes better on subset50)
    'SARIMA_111_010_52': ((1,1,1), (0,1,0,52)),  # seasonal differencing only
    'SARIMA_011_010_52': ((0,1,1), (0,1,0,52)),  # MA + seasonal differencing only
}

order_results = {}
print(f"{'Config':<22} {'AIC':>10} {'BIC':>10} {'WMAE':>10}")
print('-' * 58)

for name, (order, seasonal_order) in ORDERS_TO_TEST.items():
    try:
        m = SARIMAX(series, order=order, seasonal_order=seasonal_order,
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        fc = m.forecast(steps=len(val_series))
        w = val_set[(val_set['Store']==store) & (val_set['Dept']==dept)]['IsHoliday'].values
        wmae_score = wmae(val_series.values, fc.values, w)
        order_results[name] = {'order': order, 'seasonal_order': seasonal_order,
                               'aic': m.aic, 'bic': m.bic, 'wmae': wmae_score}
        print(f"{name:<22} {m.aic:>10.1f} {m.bic:>10.1f} {wmae_score:>10.1f}")
    except Exception as e:
        print(f"{name:<22} FAILED: {e}")

single_series_best = min(order_results, key=lambda k: order_results[k]['wmae'])
print(f"\nBest on Store 1 / Dept 1: {single_series_best} (WMAE={order_results[single_series_best]['wmae']:,.2f})")

with mlflow.start_run(run_name='SARIMA_OrderComparison_Store1_Dept1'):
    mlflow.log_param('store', 1)
    mlflow.log_param('dept', 1)
    mlflow.log_param('seasonal_period', SEASONAL_PERIOD)
    mlflow.log_param('single_series_best', single_series_best)
    for name, metrics in order_results.items():
        mlflow.log_metric(f'aic_{name}', metrics['aic'])
        mlflow.log_metric(f'wmae_{name}', metrics['wmae'])
    mlflow.log_artifact('sarima_educational_analysis.png', 'plots')
    mlflow.log_artifact('sarima_single_series_forecast.png', 'plots')

In [ ]:
#9
train_holiday = (train_set[(train_set['Store']==store) & (train_set['Dept']==dept)]
                 .set_index('Date')['IsHoliday'].asfreq('W-FRI').astype(float))
val_holiday = (val_set[(val_set['Store']==store) & (val_set['Dept']==dept)]
               .set_index('Date')['IsHoliday'].astype(float))

best_name = min(order_results, key=lambda k: order_results[k]['wmae'])
best_order = order_results[best_name]['order']
best_seasonal = order_results[best_name]['seasonal_order']

try:
    sarimax_model = SARIMAX(
        series,
        order=best_order,
        seasonal_order=best_seasonal,
        exog=train_holiday,
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False)

    sarimax_fc = sarimax_model.forecast(steps=len(val_series), exog=val_holiday)
    sarimax_score = wmae(val_series.values, sarimax_fc.values, val_holiday.values)

    baseline_score = order_results[best_name]['wmae']
    print(f"SARIMA{best_order}x{best_seasonal}      WMAE: {baseline_score:,.2f}")
    print(f"SARIMAX + IsHoliday WMAE: {sarimax_score:,.2f}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    fig.patch.set_facecolor('#0f1117')
    for ax in axes:
        ax.set_facecolor('#1a1d27')
        ax.tick_params(colors='#8b949e')

    for ax, fc, title, color in zip(
        axes,
        [forecast, sarimax_fc],
        ['SARIMA baseline', 'SARIMAX + IsHoliday'],
        ['#f78166', '#3fb950'],
    ):
        ax.plot(val_series.index, val_series.values, color='#58a6ff', linewidth=1.5, label='Actual')
        ax.plot(val_series.index, fc.values, color=color, linewidth=1.5, linestyle='--', label='Forecast')
        ax.set_title(title, color='white', fontsize=11)
        ax.legend(framealpha=0.3)

    plt.suptitle('SARIMA vs SARIMAX', color='white', fontsize=13)
    plt.tight_layout()
    plt.savefig('sarima_vs_sarimax.png', dpi=130, facecolor='#0f1117')
    plt.show()

    with mlflow.start_run(run_name='SARIMAX_Holiday_Store1_Dept1'):
        mlflow.log_params({
            'order': str(best_order),
            'seasonal_order': str(best_seasonal),
            'exog': 'IsHoliday',
            'store': 1,
            'dept': 1,
        })
        mlflow.log_metric('val_wmae', sarimax_score)
        mlflow.log_artifact('sarima_vs_sarimax.png', 'plots')
except Exception as e:
    print(f'SARIMAX failed: {e}')
    sarimax_score = None

In [ ]:
#10
def fit_sarima_series(tr_df, order, seasonal_order, exog_train=None):
    if len(tr_df) < 80:
        return None
    try:
        s = tr_df.set_index('Date')['Weekly_Sales'].asfreq('W-FRI')
        model = SARIMAX(
            s,
            order=order,
            seasonal_order=seasonal_order,
            exog=exog_train,
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit(disp=False)
        return model
    except Exception:
        return None

np.random.seed(42)
all_pairs = train.groupby(['Store', 'Dept']).size()
valid_pairs = all_pairs[all_pairs >= 80].index.tolist()
sample_pairs = [valid_pairs[i] for i in np.random.choice(len(valid_pairs), size=50, replace=False)]

def evaluate_config(config_name, order, seasonal_order, use_exog=False):
    preds, n_failed = [], 0
    print(f"\nFitting {config_name} on 50 series...")
    for store_i, dept_i in sample_pairs:
        tr = train_set[(train_set['Store']==store_i) & (train_set['Dept']==dept_i)]
        vl = val_set[(val_set['Store']==store_i) & (val_set['Dept']==dept_i)]
        if len(tr) < 80 or len(vl) == 0:
            n_failed += 1
            continue
        try:
            exog_tr = None
            exog_vl = None
            if use_exog:
                exog_tr = tr.set_index('Date')['IsHoliday'].astype(float).asfreq('W-FRI')
                exog_vl = vl.set_index('Date')['IsHoliday'].astype(float)
            model = fit_sarima_series(tr, order, seasonal_order, exog_tr)
            if model is None:
                n_failed += 1
                continue
            fc = np.clip(model.forecast(steps=len(vl), exog=exog_vl).values, 0, None)
            for j, (_, row) in enumerate(vl.iterrows()):
                if j < len(fc):
                    preds.append({
                        'Actual': row['Weekly_Sales'],
                        'Predicted': fc[j],
                        'IsHoliday': row['IsHoliday'],
                    })
        except Exception:
            n_failed += 1

    if not preds:
        return None
    pred_df = pd.DataFrame(preds)
    score = wmae(pred_df['Actual'].values, pred_df['Predicted'].values, pred_df['IsHoliday'].values)
    print(f"  WMAE: {score:,.2f}  (failed: {n_failed})")

    with mlflow.start_run(run_name=config_name):
        mlflow.log_params({
            'order': str(order),
            'seasonal_order': str(seasonal_order),
            'exog': 'IsHoliday' if use_exog else 'none',
            'n_series': len(sample_pairs),
            'min_series_length': 80,
        })
        mlflow.log_metric('val_wmae', score)
        mlflow.log_metric('n_failed', n_failed)
    return score

subset_results = {}
for name, (order, seasonal_order) in ORDERS_TO_TEST.items():
  score = evaluate_config(f'SARIMA_Subset50_{name}', order, seasonal_order, use_exog=False)
  if score is not None:
      subset_results[name] = score

sarimax_subset_score = evaluate_config(
    'SARIMAX_Holiday_Subset50',
    best_order,
    best_seasonal,
    use_exog=True,
)
if sarimax_subset_score is not None:
    subset_results['SARIMAX_IsHoliday'] = sarimax_subset_score

In [ ]:
#11
import json

if 'subset_results' not in globals() or not subset_results:
    raise NameError(
        "subset_results is missing. Run cell #10 first, then run this cell again."
    )

all_scores = {k: v for k, v in subset_results.items() if v is not None}

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d27')

names = list(all_scores.keys())
scores = list(all_scores.values())
colors = ['#8b949e'] * len(names)
colors[int(np.argmin(scores))] = '#f78166'

bars = ax.bar(names, scores, color=colors, edgecolor='none', width=0.5)
ax.bar_label(bars, labels=[f'{v:,.1f}' for v in scores], padding=5, color='white', fontsize=10)
ax.set_title('SARIMA subset50 comparison (WMAE, lower is better)', color='white', fontsize=13)
ax.set_ylabel('WMAE', color='#8b949e')
ax.tick_params(colors='#8b949e', axis='x', rotation=25)
plt.tight_layout()
plt.savefig('sarima_all_runs.png', dpi=130, facecolor='#0f1117')
plt.show()

best_run_name = names[int(np.argmin(scores))]
best_subset_wmae = min(scores)

if best_run_name == 'SARIMAX_IsHoliday':
    best_use_exog = True
    best_model_order = best_order
    best_model_seasonal = best_seasonal
elif best_run_name in ORDERS_TO_TEST:
    best_use_exog = False
    best_model_order, best_model_seasonal = ORDERS_TO_TEST[best_run_name]
else:
    raise ValueError(f"Unknown best config: {best_run_name}")

print(f"Best on subset50: {best_run_name}")
print(f"Order: {best_model_order} | Seasonal: {best_model_seasonal} | Uses exog: {best_use_exog}")
print(f"Subset50 WMAE: {best_subset_wmae:,.2f}")
print(f"Single-series best was: {single_series_best} (WMAE={order_results[single_series_best]['wmae']:,.2f})")

if best_use_exog:
    best_model = SARIMAX(
        series,
        order=best_model_order,
        seasonal_order=best_model_seasonal,
        exog=train_holiday,
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False)
else:
    best_model = SARIMAX(
        series,
        order=best_model_order,
        seasonal_order=best_model_seasonal,
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False)

# Pipeline spec logged as JSON (statsmodels SARIMAX cannot be wrapped in sklearn Pipeline for MLflow)
pipeline_spec = {
    'preprocess': {
        'markdown_imputation': 'fill_zero',
        'negative_sales': 'clip_to_zero',
        'sort_by': ['Store', 'Dept', 'Date'],
        'frequency': 'W-FRI',
    },
    'model': {
        'type': 'SARIMAX',
        'order': list(best_model_order),
        'seasonal_order': list(best_model_seasonal),
        'exog': 'IsHoliday' if best_use_exog else None,
        'demo_series': {'store': int(store), 'dept': int(dept)},
    },
    'selection': {
        'best_config': best_run_name,
        'criterion': 'subset50_val_wmae',
        'single_series_best': single_series_best,
    },
}
with open('sarima_best_pipeline_spec.json', 'w', encoding='utf-8') as f:
    json.dump(pipeline_spec, f, indent=2)

with mlflow.start_run(run_name='SARIMA_Best'):
    mlflow.log_param('best_config', best_run_name)
    mlflow.log_param('best_order', str(best_model_order))
    mlflow.log_param('best_seasonal_order', str(best_model_seasonal))
    mlflow.log_param('best_uses_exog', best_use_exog)
    mlflow.log_param('single_series_best', single_series_best)
    mlflow.log_param('n_series_evaluated', len(sample_pairs))
    mlflow.log_param('note', 'statsmodels model registered; pipeline spec saved as JSON artifact')
    mlflow.log_metric('best_subset50_wmae', best_subset_wmae)
    mlflow.log_metric('single_series_best_wmae', order_results[single_series_best]['wmae'])
    mlflow.log_artifact('sarima_all_runs.png', 'plots')
    mlflow.log_artifact('sarima_best_pipeline_spec.json', 'pipeline')

    mlflow.statsmodels.log_model(
        statsmodels_model=best_model,
        artifact_path='sarima-best-model',
        registered_model_name='SARIMA_Walmart_Best',
    )
    print('Best SARIMA model registered in MLflow Model Registry')